# Twinkle DPO 训练

> 🎯 **训练目标**：通过 DPO 偏好优化，让模型学会用 **中文 + Emoji 风格** 回答问题，生成更生动友好的回复。

本 Notebook 演示如何使用 **DPO（Direct Preference Optimization，直接偏好优化）** 算法通过 Twinkle 对语言模型进行微调。

### 什么是 DPO？

DPO 是一种从人类偏好数据中直接学习的训练方法，无需显式奖励模型。训练流程如下：
1. 准备 **偏好数据集**，每条样本包含一个 prompt，以及一条「更好的回答」（chosen）和一条「较差的回答」（rejected）
2. 用 **参考模型**（base model，禁用 LoRA）对 chosen/rejected 做前向推理，得到参考 log 概率 `ref_logps`
3. 将 `ref_logps` 附加到训练数据中，再用 **策略模型**（带 LoRA）做前向反向传播，计算 DPO 损失
4. 执行 **优化器更新**，让模型更倾向于生成 chosen 回答

### 整体流程

```
准备数据集 → 创建 LoRA 训练客户端 → 训练循环：
  参考前向（disable_lora=True）→ 附加 ref_logps → DPO 前向反向 → 优化器步骤
```

### DPO 批次格式

为了让每个 DP（Data Parallel）分片都含有完整的 chosen/rejected 对，批次采用**交错排列**格式：

```
[pos_1, neg_1, pos_2, neg_2, ..., pos_N, neg_N]
```

### 前置条件

| 条件 | 说明 |
|------|------|
| 环境变量 | 设置 `MODELSCOPE_TOKEN` |
| 依赖安装 | `pip install twinkle-kit[tinker]` |

> 💡 **获取 Token**：访问 [ModelScope Token 页面](https://www.modelscope.cn/my/access/token) 获取你的 `MODELSCOPE_TOKEN`，并设置为环境变量：`export MODELSCOPE_TOKEN=<你的Token>`


## 🚀 零卡训练服务化（Serverless Training）

本 Notebook 运行在 **ModelScope 零卡训练平台** 上。你无需自备 GPU，只需在 Notebook 中编写训练逻辑，平台会自动调度云端 GPU 资源完成训练。

### 架构示意图

```
┌─────────────────────────────────────────────────────────────┐
│                    你的 Notebook（CPU 环境）                   │
│                                                             │
│   ┌──────────┐    HTTP / gRPC     ┌──────────────────────┐  │
│   │  Twinkle │ ─────────────────► │   ModelScope 云端     │  │
│   │  Client  │ ◄───────────────── │   GPU 训练集群        │  │
│   └──────────┘    训练结果返回      │                      │  │
│       │                           │  ┌────┐ ┌────┐ ┌────┐│  │
│       │  构造数据                   │  │GPU0│ │GPU1│ │... ││  │
│       │  发送训练请求               │  └────┘ └────┘ └────┘│  │
│       │  接收指标/检查点            │  模型加载 + LoRA 训练  │  │
│       ▼                           └──────────────────────┘  │
│   ┌──────────┐                                              │
│   │ 数据准备  │  Dataset / DataLoader / Preprocessor         │
│   └──────────┘                                              │
└─────────────────────────────────────────────────────────────┘
```

### 核心优势

| 特性 | 说明 |
|------|------|
| **零卡启动** | Notebook 本身不需要 GPU，训练在云端自动执行 |
| **按需付费** | 仅在训练时占用 GPU 资源 |
| **开箱即用** | 预置主流模型，无需下载权重 |
| **LoRA 微调** | 高效参数微调，几分钟即可完成小规模训练 |

> 🔗 本项目由 [Twinkle](https://github.com/modelscope/twinkle) 框架提供支持 | [GitHub](https://github.com/modelscope/twinkle)

## 第一步：导入依赖与全局配置

| 配置项 | 默认值 | 说明 |
|--------|--------|------|
| `BASE_MODEL` | Qwen/Qwen3.6-35B-A3B | 基座模型 |
| `BATCH_SIZE` | 4 | 每步处理的 DPO 样本对数 |
| `LEARNING_RATE` | 1e-4 | 学习率 |
| `DPO_BETA` | 0.1 | DPO 温度系数，控制偏好强度 |
| `SFT_WEIGHT` | 1.0 | SFT 损失权重（辅助监督信号） |
| `MAX_LENGTH` | 2048 | 序列最大长度 |
| `LORA_RANK` | 8 | LoRA 秩 |
| `DATA_NUM` | 5000 | 使用的数据集样本数量 |

## 环境安装

首次运行前，请先执行以下安装命令。如已安装可跳过此步。

In [ ]:
!pip install twinkle-kit[tinker] -q

In [ ]:
import os
import numpy as np
import torch
from tqdm import tqdm
from typing import Any, Dict, List


from tinker import types
from twinkle import init_tinker_client, get_logger
from twinkle.dataset import Dataset, DatasetMeta, LazyDataset
from twinkle.dataloader import DataLoader
from twinkle.preprocessor import EmojiDPOProcessor
from twinkle.server.common import input_feature_to_datum

logger = get_logger()

# ========== 全局配置 ==========
BASE_MODEL    = 'Qwen/Qwen3.6-35B-A3B'
BASE_URL      = 'http://www.modelscope.cn/twinkle'
API_KEY       = 'EMPTY_API_KEY'
DATASET_ID    = 'ms://hjh0119/shareAI-Llama3-DPO-zh-en-emoji'

BATCH_SIZE    = 4
LEARNING_RATE = 1e-4
DPO_BETA      = 0.1
SFT_WEIGHT    = 1.0
MAX_LENGTH    = 2048
LORA_RANK     = 8
DATA_NUM      = 5000
SYSTEM_PROMPT = 'You are a helpful assistant.'

## 第二步：准备数据集

本示例使用 ModelScope 上的 `shareAI-Llama3-DPO-zh-en-emoji` 数据集，格式为：

```json
{
  "prompt": "问题文本",
  "answer_zh": "中文回答（chosen）",
  "answer_en": "英文回答（rejected）"
}
```

`EmojiDPOProcessor` 会将每条原始样本转换为：
- `positive`：包含 chosen 回答的 `Trajectory`
- `negative`：包含 rejected 回答的 `Trajectory`

`dataset.encode()` 会自动识别 `positive`/`negative` 格式，将双轨迹编码为对应的 `InputFeature`。

> **注意**：使用 `LazyDataset` 可以在迭代时按需加载数据，节省内存。

In [ ]:
def create_dpo_dataset():
    """创建 DPO 数据集，返回包含 positive/negative 对的编码数据集。"""
    dataset = LazyDataset(DatasetMeta(DATASET_ID, data_slice=range(DATA_NUM)))
    dataset.set_template('Qwen3_5Template', model_id=f'ms://{BASE_MODEL}', max_length=MAX_LENGTH)
    dataset.map(
        EmojiDPOProcessor,
        init_args={'system': SYSTEM_PROMPT},
    )
    # EmojiDPOProcessor 返回 {'positive': InputFeature, 'negative': InputFeature}
    # encode 会自动处理该格式
    dataset.encode()
    return dataset


dataset = create_dpo_dataset()
dataloader = DataLoader(dataset=dataset, batch_size=BATCH_SIZE)
print(f'数据集大小: {len(dataset)} 条')
print(f'每轮训练步数: {len(dataloader)} 步')

## 第三步：构造 DPO 批次

DPO 要求每个训练批次同时包含 chosen 和 rejected 样本，且必须**成对出现**。

为确保 Data Parallel 切分后每个设备都能看到完整的 chosen/rejected 对，我们将批次重排为交错格式：

```
原始批次：[{positive: A+, negative: A-}, {positive: B+, negative: B-}]
交错后：  [A+, A-, B+, B-]
```

这样无论按多少个 DP 分片切割，每个分片都包含完整的偏好对。

In [ ]:
def prepare_dpo_batch(batch: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """将批次重排为交错格式 [pos_1, neg_1, pos_2, neg_2, ...]。

    Args:
        batch: 原始批次，每条含 'positive' 和 'negative' InputFeature。

    Returns:
        交错后的列表，保证每个 DP 切片包含完整的 chosen/rejected 对。
    """
    result = []
    for row in batch:
        base_fields = {k: v for k, v in row.items() if k not in ('positive', 'negative')}
        pos_sample  = {**base_fields, **row['positive']}
        neg_sample  = {**base_fields, **row['negative']}
        result.append(pos_sample)
        result.append(neg_sample)
    return result


# 验证：取第一个 batch 演示交错逻辑
sample_batch = next(iter(dataloader))
dpo_batch = prepare_dpo_batch(sample_batch)
print(f'原始批次大小: {len(sample_batch)} 对')
print(f'交错后批次大小: {len(dpo_batch)} 条（每对展开为 2 条）')

## 第四步：初始化客户端

### 客户端初始化顺序

> **重要**：必须先调用 `init_tinker_client()` 完成 Tinker 运行时初始化，再 import `ServiceClient`。

### LoRA 训练客户端

`create_lora_training_client` 会在服务端创建一个携带 LoRA 适配器的训练会话：
- `base_model`：指定基座模型（需与服务端已加载模型一致）
- `rank`：LoRA 秩，秩越大表达能力越强，但显存消耗也越大


In [ ]:
# 必须先初始化 Tinker 客户端，再 import ServiceClient
init_tinker_client()

from tinker import ServiceClient  # noqa: E402

service_client = ServiceClient(base_url=BASE_URL, api_key=API_KEY)
training_client = service_client.create_lora_training_client(
    base_model=BASE_MODEL,
    rank=LORA_RANK,
)
logger.info(f'LoRA 训练客户端已创建（rank={LORA_RANK}）')
logger.info(f'开始 DPO 训练：beta={DPO_BETA}, lr={LEARNING_RATE}')

## 第五步：DPO 训练主循环

每个训练步骤包含四个阶段：

### 5.1 参考前向传播（Reference Forward）

调用 `training_client.forward()` 并设置 `disable_lora=True`，让服务端以**纯基座模型**进行前向计算：
- LoRA 权重不参与计算图 → 反向传播产生零梯度，**不影响 LoRA 参数**
- 返回每个 token 的 log 概率 `ref_logps`，作为 DPO 损失的参考基线

### 5.2 附加参考 Log 概率

将参考前向的输出 `ref_logps` 附加到每条 Datum 的 `loss_fn_inputs` 中。
服务端检测到 `ref_logps` 字段后，会自动切换为 `DPOLoss + DPOMetric`。

### 5.3 DPO 前向反向传播

调用 `training_client.forward_backward()` 使用 `importance_sampling` 损失：
- `dpo_beta`：控制策略偏离参考模型的程度，越大越保守
- `dpo_sft_weight`：SFT 辅助损失权重，有助于稳定训练

### 5.4 优化器步骤

`optim_step` 更新 LoRA 参数，并返回 DPO 相关指标（chosen/rejected reward、reward margin 等）。

> **提示**：DPO 训练中 `reward_margin`（chosen reward - rejected reward）持续增大，说明模型正在正确学习偏好。

In [ ]:
for step, batch in tqdm(enumerate(dataloader), total=len(dataloader)):

    # ---- 数据预处理：numpy/torch → Python list（序列化需要）----
    for row in batch:
        for key in list(row.keys()):
            if isinstance(row[key], np.ndarray):
                row[key] = row[key].tolist()
            elif isinstance(row[key], torch.Tensor):
                row[key] = row[key].cpu().numpy().tolist()

    # 将批次重排为交错格式 [pos, neg, pos, neg, ...]
    dpo_batch = prepare_dpo_batch(batch)

    # 将每条 InputFeature dict 转换为 Tinker Datum
    input_datums = [input_feature_to_datum(row) for row in dpo_batch]

    # =================================================================
    # A. 参考前向传播（base model，disable_lora=True）
    #    LoRA 不在计算图中 → 反向不更新 LoRA，安全执行
    # =================================================================
    ref_result = training_client.forward(
        input_datums,
        'cross_entropy',
        loss_fn_config={'disable_lora': True},
    ).result()

    # =================================================================
    # B. 将参考 log 概率附加到每条 Datum 的 loss_fn_inputs
    #    服务端检测到 ref_logps 后自动切换 DPOLoss + DPOMetric
    # =================================================================
    for datum, ref_out in zip(input_datums, ref_result.loss_fn_outputs):
        ref_logprobs_np = np.array(ref_out['logprobs'].tolist(), dtype=np.float32)
        datum.loss_fn_inputs['ref_logps'] = types.TensorData.from_numpy(ref_logprobs_np)

    # =================================================================
    # C. DPO 前向反向传播
    #    服务端自动计算 DPO 损失，无需手动实现
    # =================================================================
    fwdbwd_result = training_client.forward_backward(
        input_datums,
        'importance_sampling',
        loss_fn_config={
            'dpo_beta':       DPO_BETA,
            'dpo_sft_weight': SFT_WEIGHT,
        },
    ).result()

    # =================================================================
    # D. 优化器步骤
    #    更新 LoRA 参数，DPOMetric 在服务端自动计算并随结果返回
    # =================================================================
    optim_result = training_client.optim_step(
        types.AdamParams(learning_rate=LEARNING_RATE)
    ).result()

    logger.info(f'[Step {step}] metrics={optim_result.metrics}')


## 第六步：保存与导出检查点

训练完成后，使用 `save_state` 将 LoRA 权重保存到服务端：
- 保存路径由服务端配置决定，返回的 `save_result.path` 为实际路径
- 该检查点包含 LoRA 适配器权重，可直接加载到基座模型上进行推理

### 可选：上传到 ModelScope Hub

取消注释下方代码，可将检查点一键发布到 ModelScope 模型库，方便共享和部署。

In [ ]:
# 保存 LoRA 检查点
save_result = training_client.save_state('dpo-lora-final').result()
logger.info(f'检查点已保存：{save_result.path}')

# （可选）上传到 ModelScope Hub
# YOUR_USER_NAME = 'your_username'
# hub_model_id = f'{YOUR_USER_NAME}/twinkle-tinker-dpo-lora'
# training_client.publish_checkpoint_from_tinker_path(save_result.path).result()
# logger.info(f'检查点已上传至 Hub: {hub_model_id}')

## 推理（Inference）

训练完成后，可以直接使用 **线上服务** 进行推理，无需本地 GPU。

通过 `save_weights_and_get_sampling_client` 或 `create_sampling_client` 加载训练好的 LoRA 检查点，即可在线采样生成。

> 将下方 `weight_path` 替换为训练输出的检查点路径（`twinkle://...` 格式）。

In [ ]:
# 推理示例（使用线上服务，无需本地 GPU）
import os
from tinker import types
from twinkle import init_tinker_client, get_logger
from twinkle.data_format import Message, Trajectory
from twinkle.template import Template

logger = get_logger()

BASE_MODEL = 'Qwen/Qwen3.6-35B-A3B'

# TODO: 替换为训练输出的检查点路径
weight_path = '<替换为你的 twinkle:// 检查点路径>'  # 例如: save_result.path

init_tinker_client()
from tinker import ServiceClient

service_client = ServiceClient(
    base_url='http://www.modelscope.cn/twinkle',
    api_key=os.environ.get('MODELSCOPE_TOKEN'),
)

# 加载 LoRA 检查点并创建采样客户端
sampling_client = service_client.create_sampling_client(
    model_path=weight_path,
    base_model=BASE_MODEL,
)

# 构造 Prompt
template = Template(model_id=f'ms://{BASE_MODEL}')
trajectory = Trajectory(
    messages=[
        Message(role='system', content='You are a helpful assistant.'),
        Message(role='user', content='你好，请介绍一下你自己。'),
    ]
)

input_feature = template.encode(trajectory, add_generation_prompt=True)
input_ids = input_feature['input_ids'].tolist()

# 采样
prompt = types.ModelInput.from_ints(input_ids)
params = types.SamplingParams(max_tokens=256, temperature=0.7)

print('Sampling...')
future = sampling_client.sample(prompt=prompt, sampling_params=params, num_samples=3)
result = future.result()

# 输出结果
print('Responses:')
for i, seq in enumerate(result.sequences):
    print(f'{i}: {repr(template.decode(seq.tokens))}')

## 合并权重并导出

训练得到的 LoRA 权重可以与原始模型合并，导出为完整的 HuggingFace 模型，方便后续部署和推理。

> **注意**：合并操作需要 GPU 资源（需要加载完整模型），请在有足够显存的环境下执行。

```bash
CUDA_VISIBLE_DEVICES=0,1,2,3 \
NPROC_PER_NODE=4 \
/opt/conda/envs/twinkle/bin/megatron export \
    --model Qwen/Qwen3.6-35B-A3B \
    --adapters <替换为你的 LoRA 检查点路径> \
    --output_dir <替换为输出目录> \
    --merge_lora true \
    --to_hf true \
    --tensor_model_parallel_size 2 \
    --expert_model_parallel_size 2 \
    --pipeline_model_parallel_size 2
```

**参数说明**：

| 参数 | 说明 |
|------|------|
| `--model` | 基座模型 ID |
| `--adapters` | 训练保存的 LoRA 检查点路径 |
| `--output_dir` | 合并后的完整模型输出目录 |
| `--merge_lora true` | 将 LoRA 权重合并到基座模型中 |
| `--to_hf true` | 导出为 HuggingFace 格式 |
| `--tensor_model_parallel_size` | 张量并行大小 |
| `--expert_model_parallel_size` | 专家并行大小（MoE 模型专用） |
| `--pipeline_model_parallel_size` | 流水线并行大小 |

合并完成后，输出目录中即为完整的 HuggingFace 模型，可直接用于推理或部署。

## 总结

本 Notebook 展示了使用 Twinkle 进行 DPO 训练的完整流程：

| 步骤 | 操作 | 关键 API |
|------|------|----------|
| 数据准备 | 加载偏好数据集，用 `EmojiDPOProcessor` 生成 chosen/rejected 对 | `LazyDataset`, `EmojiDPOProcessor` |
| 批次构造 | 交错排列 `[pos, neg, pos, neg, ...]` | `prepare_dpo_batch` |
| 参考前向 | 用基座模型计算 `ref_logps` | `training_client.forward(..., disable_lora=True)` |
| DPO 训练 | 附加 `ref_logps`，自动触发 DPO 损失计算 | `training_client.forward_backward(..., 'importance_sampling')` |
| 参数更新 | 更新 LoRA，获取 DPO 指标 | `training_client.optim_step(AdamParams)` |
| 保存导出 | 保存 LoRA 检查点 | `training_client.save_state` |

### 调参建议

- **`DPO_BETA`**：通常在 0.01 ~ 0.5 之间，越小越激进，越大越保守
- **`SFT_WEIGHT`**：增大该值可防止模型忘记基础能力（灾难性遗忘）
- **`LORA_RANK`**：rank=8 适合大多数场景；数据量大或任务复杂时可尝试 16/32